In [7]:
import torch
from transformers import RobertaTokenizer,RobertaModel,RobertaForMaskedLM, pipeline, BertForMaskedLM, BertTokenizer, BertModel, AlbertConfig, AlbertModel, AlbertTokenizer, AlbertForMaskedLM , TFAutoModel, AutoModelForMaskedLM, AutoTokenizer, AutoModelForCausalLM
import math
import numpy as np
import json
import random
import pandas as pd
import tensorflow as tf
from torch.nn import CrossEntropyLoss
from tqdm.notebook import tqdm
import re
from scipy.optimize import linear_sum_assignment

import SMI_Methods

In [8]:
import logging
import os
import sys
logging.basicConfig(level=logging.INFO)
import warnings
warnings.filterwarnings('ignore')

In [10]:
# Dictionary to map Slovak characters to their Croatian equivalents.
SLOVAK_TO_CROATIAN = {
    'á': 'a',
    'ä': 'a',
    'ď': 'đ',
    'é': 'e',
    'í': 'i',
    'ľ': 'l',
    'ĺ': 'l',
    'ň': 'nj',
    'ó': 'o',
    'ô': 'o',
    'ŕ': 'r',
    'ť': 'ć',
    'ú': 'u',
    'ý': 'j',
    'ä': 'a',
}

CROATIAN_TO_SLOVAK = {
    'lj': 'ľ',
    'nj': 'ň',
    'ć': 'č',
    'đ': 'ď',
    'dž': 'ǧ',
}

def is_list(element):
    """Check if the given element is a list."""
    return isinstance(element, list)

def slovak_to_croatian(text):
    """Transcribes a Slovak text into Croatian phonetically."""
    for slovak_char, croatian_char in SLOVAK_TO_CROATIAN.items():
        text = text.replace(slovak_char, croatian_char)
        text = text.replace(slovak_char.upper(), croatian_char.upper())  # Replace uppercase characters.
    return text

def croatian_to_slovak(text):
    """Transcribes a Croatian text into Slovak phonetically."""
    # We need to replace longer substrings first to avoid partial matches.
    for croatian_char, slovak_char in sorted(CROATIAN_TO_SLOVAK.items(), key=lambda x: -len(x[0])):
        text = text.replace(croatian_char, slovak_char)
        text = text.replace(croatian_char.upper(), slovak_char.upper())  # Replace uppercase characters.
    return text

def replace_in_nested_list(nested_list, direction="stoc"):
    """
    Perform the replacement operation on the deepest sublists of arbitrary depth.
    :param nested_list: The nested list to operate on.
    :param direction: The direction of the transcription ("stoc" for Slovak to Croatian,
                      "ctos" for Croatian to Slovak).
    :return: The nested list with replaced strings.
    """
    if not is_list(nested_list):
        return slovak_to_croatian(nested_list) if direction == "stoc" else croatian_to_slovak(nested_list)
    return [replace_in_nested_list(sublist, direction) for sublist in nested_list]


In [4]:
def see_tokens(model, tokenizer, data):
    r = 0
    for d in data:
        for s in d:
            for l in s:
                r += len(tokenizer.tokenize(l))
    return r

In [5]:
def run_tests(model, tokenizer, lang, transliterate):
    word_list = SMI_Methods.prep_words(lang)
    data = SMI_Methods.prep_data(lang, word_list)
    
       
    word_list = replace_in_nested_list(word_list, direction = transliterate)
    data = replace_in_nested_list(data, direction = transliterate)
    lang = transliterate
    
    token_count = see_tokens(model, tokenizer, data)
    
    results = []
    for desc, d in zip(["1 Line Data", "3 Line Data", "3 Line Data (unfilled)"], [data[0], data[1], data[2]]):
        scores_line = [0,0,0,0]
        for i in range(4):
            sc = SMI_Methods.score_model(model, tokenizer, d[i], word_list[i])
            scores_line[0] += sc[0]
            scores_line[1] += sc[1]
            scores_line[2] += sc[2]
            scores_line[3] += sc[3]
        results.append({
            "Language": lang,
            "Description": desc,
            "Token Count": token_count,
            "Top 1 Score": scores_line[0]/4,
            "Top 3 Score": scores_line[1]/4,
            "COS": scores_line[2]/4,
            "GAS": scores_line[3]/4
        })

    df = pd.DataFrame(results)
    return df


In [11]:
tokenizer = AutoTokenizer.from_pretrained("Milos/slovak-gpt-j-162M")
model = AutoModelForCausalLM.from_pretrained("Milos/slovak-gpt-j-162M").cuda()
df = run_tests(model, tokenizer, "Croatian", "ctos")
df.to_csv("Outputs/Transliterated_CrtoS.csv")

  0%|          | 0/12 [00:00<?, ?it/s]

  0%|          | 0/12 [00:00<?, ?it/s]

  0%|          | 0/12 [00:00<?, ?it/s]

  0%|          | 0/12 [00:00<?, ?it/s]

  0%|          | 0/12 [00:00<?, ?it/s]

...

  0%|          | 0/12 [00:00<?, ?it/s]

  0%|          | 0/12 [00:00<?, ?it/s]

  0%|          | 0/12 [00:00<?, ?it/s]

  0%|          | 0/12 [00:00<?, ?it/s]

  0%|          | 0/12 [00:00<?, ?it/s]

In [12]:
tokenizer = AutoTokenizer.from_pretrained("macedonizer/hr-gpt2")
model = AutoModelForCausalLM.from_pretrained("macedonizer/hr-gpt2").cuda()
df = run_tests(model, tokenizer, "Slovak","stoc")
df.to_csv("Outputs/Transliterated_StoCr.csv")

Special tokens have been added in the vocabulary, make sure the associated word embeddings are fine-tuned or trained.


  0%|          | 0/12 [00:00<?, ?it/s]

  0%|          | 0/12 [00:00<?, ?it/s]

  0%|          | 0/12 [00:00<?, ?it/s]

  0%|          | 0/12 [00:00<?, ?it/s]

...

  0%|          | 0/12 [00:00<?, ?it/s]

  0%|          | 0/12 [00:00<?, ?it/s]

  0%|          | 0/12 [00:00<?, ?it/s]

  0%|          | 0/12 [00:00<?, ?it/s]

  0%|          | 0/12 [00:00<?, ?it/s]